# 1. `CIFUtils` - as a USER


## 1.1 Loading from `mmCIF`

In [ ]:
from cifutils import parse
from pathlib import Path

# ...define the path to the CIF file
# (.cif, .bcif, .cif.gz, and .bcif.gz files are supported)
path = "/projects/ml/frozen_pdb_copies/2024_12_01_pdb/ne/3ne2.cif.gz"

# ...parse the CIF file, enumerating non-default parameters
# (See `parser.parse` documentation within `cifutils` for more details)
result_dict = parse(
    filename=Path(path),
    build_assembly=["1"],
    add_missing_atoms=True,
    remove_waters=True,
    hydrogen_policy="remove",
    model=1,
)

# ...print the keys of the resulting dictionary
print("Keys:")
print(list(result_dict.keys()))

In [ ]:
from cifutils.utils.visualize import view

asym_unit = result_dict["asym_unit"][0]
asym_unit = asym_unit[asym_unit.occupancy > 0]

# ..visualize the asymmetric unit
view(asym_unit)

In [ ]:
assembly = result_dict["assemblies"]["1"][0]
assembly = assembly[assembly.occupancy > 0]

# ...visualize the assembly
view(assembly)

In [ ]:
# ...print the chain information
# (A list of dictionaries, each containing information about a chain)
result_dict["chain_info"]

In [ ]:
# ...print the ligand information
# (A list of dictionaries, each containing information about a ligand. For example, the ligand of interest, or the ligand fit-to-density)
result_dict["ligand_info"]

In [ ]:
# ...print the metadata
result_dict["metadata"]

In [ ]:
# ...print the annotation categories
print(result_dict["asym_unit"][0].get_annotation_categories())

# ...and take a peak at what an AtomArray looks like
result_dict["asym_unit"][0]

## 1.2 Manipulating the `AtomArray`

In [ ]:
# ...extract the first assembly, first model
assembly = result_dict["assemblies"]["1"][0]

# ...get the coordinates of all of the resolved CA atoms
print("Coordinates of all resolved CA atoms:")
ca = assembly[(assembly.atom_name == "CA") & (assembly.occupancy > 0)]
print(ca.coord.shape)

# ...get the coordinates of chain A
print("Coordinates of chain A (all heavy atoms):")
chain = assembly[assembly.chain_id == "A"]
print(chain.coord.shape)

In [ ]:
# ...check bonds
# For information about bonds in `biotite`, see: https://www.biotite-python.org/latest/apidoc/biotite.structure.BondList.html
assembly.bonds.as_array()

## 1.3 Basic distance computations

In [ ]:
import biotite.structure as struc
import numpy as np

### ATOM TO ATOM
# Distance between two atoms
distance = struc.distance(ca.coord[0], ca.coord[1])  # Distance between first two C-alpha atoms
print(f"Distance between first two C-alpha atoms: {distance:.2f} Å")

In [ ]:
### ARRAY TO ATOM
# Distance between one atom and all other atoms in the array
distance = struc.distance(ca[0], ca)
print(f"Distances between first C-alpha atom and all other C-alpha atoms: {distance}")

## 1.4 `CellList` - Efficient distance computations

In [12]:
# ...filter to resolved atoms
resolved_atom_array = assembly[assembly.occupancy > 0]

# ...build CellList
cell_list = struc.CellList(resolved_atom_array, cell_size=5.0)

In [ ]:
### ATOMS NEAR ATOM
# Get all atoms within 5 Å of the first atom
print(
    f"Target atom: {resolved_atom_array[0].coord} {resolved_atom_array[0].atom_name} {resolved_atom_array[0].res_name} {resolved_atom_array[0].res_id}"
)
near_atoms = cell_list.get_atoms(resolved_atom_array[0].coord, radius=4)
print(f"Number of atoms within 7 Å of the first atom: {near_atoms.shape[0]}")
print(f"Atom indices: {near_atoms}")
chain_ids = resolved_atom_array.chain_id[near_atoms]
print(f"Chain IDs: {chain_ids}")
residue_ids = resolved_atom_array.res_id[near_atoms]
print(f"Residue IDs: {residue_ids}")
print(f"Residue names: {resolved_atom_array.res_name[near_atoms]}")

# 2. `Datahub` - as a USER

## 2.1 Pre-built PDB datasets on disk

In [ ]:
import pandas as pd

# ...load the dataframes from the parquet files
PN_UNITS_DF = pd.read_parquet("/home/ncorley/projects/datahub/tests/data/pn_units_df.parquet")
INTERFACES_DF = pd.read_parquet("/home/ncorley/projects/datahub/tests/data/interfaces_df.parquet")

# For the paths to the FULL parquets (the above are a sample), run:
# NOTE: These are loading large parquet files, so you may must subset the columns to load to avoid memory issues

# _SHARED_COLUMNS_TO_LOAD = ["example_id", "pdb_id", "assembly_id", "method"]
# PN_UNITS_COLUMNS_TO_LOAD = _SHARED_COLUMNS_TO_LOAD + [
#     "q_pn_unit_iid",
#     "q_pn_unit_close_pn_unit_iids",
#     "q_pn_unit_type",
#     "q_pn_unit_non_polymer_res_names",
# ]
# INTERFACES_COLUMNS_TO_LOAD = _SHARED_COLUMNS_TO_LOAD + [
#     "pn_unit_1_iid",
#     "pn_unit_2_iid",
#     "close_pn_unit_iids",
#     "pn_unit_1_type",
#     "pn_unit_2_type",
#     "pn_unit_1_non_polymer_res_names",
#     "pn_unit_2_non_polymer_res_names",
# ]

# PN_UNITS_DF_FULL = pd.read_parquet(
#     "/projects/ml/RF2_allatom/data_preprocessing/PDB_2024_09_10/pn_units_df.parquet", columns=PN_UNITS_COLUMNS_TO_LOAD
# )
# INTERFACES_DF_FULL = pd.read_parquet(
#     "/projects/ml/RF2_allatom/data_preprocessing/PDB_2024_09_10/interfaces_df.parquet",
#     columns=INTERFACES_COLUMNS_TO_LOAD,
# )

# ...as an example, print the first few rows of the PN_UNITS_DF dataframe
PN_UNITS_DF

In [ ]:
print(PN_UNITS_DF.columns)

In [ ]:
from cifutils.enums import ChainType

# ...get all RNA examples in the PDB
PN_UNITS_DF_RNA = PN_UNITS_DF[PN_UNITS_DF.q_pn_unit_type == ChainType.RNA.value]
print("RNA Dataframe:", PN_UNITS_DF_RNA.shape)

# ...get all examples with covalent modifications
PN_UNITS_DF_COVALENT_MODIFICATIONS = PN_UNITS_DF[PN_UNITS_DF.q_pn_unit_bonded_polymer_pn_units != "set()"]
print("Covalent Modifications Dataframe:", PN_UNITS_DF_COVALENT_MODIFICATIONS.shape)

In [ ]:
from datahub.datasets.parsers import InterfacesDFParser, PNUnitsDFParser
from datahub.datasets.datasets import PandasDataset, ConcatDatasetWithID, StructuralDatasetWrapper
from datahub.pipelines.rf2aa import build_rf2aa_transform_pipeline

# ...define the paths to the MSA directories
PROTEIN_MSA_DIRS = [
    {"dir": "/projects/msa/rf2aa_af3/rf2aa_paper_model_protein_msas", "extension": ".a3m.gz", "directory_depth": 2},
    {
        "dir": "/projects/msa/rf2aa_af3/missing_msas_through_2024_08_12",
        "extension": ".msa0.a3m.gz",
        "directory_depth": 2,
    },
]
RNA_MSA_DIRS = [
    {"dir": "/projects/msa/rf2aa_af3/rf2aa_paper_model_rna_msas", "extension": ".afa", "directory_depth": 0}
]

# ...define the filters to apply to the dataframes
SHARED_TEST_FILTERS = [
    "deposition_date < '2022-01-01'",
    "resolution < 5.0 and ~method.str.contains('NMR')",
    "num_polymer_pn_units <= 20",  # To ensure we don't freeze loading a massive example
    "cluster.notnull()",
    "method in ['X-RAY_DIFFRACTION', 'ELECTRON_MICROSCOPY']",
]

# Define the PDB datasets with their respective parsers...
RF2AA_PN_UNITS_DATASET = StructuralDatasetWrapper(
    dataset_parser=PNUnitsDFParser(),  # This is the parser for the PNUnitsDF dataframe; it will be used to extract the data from the dataframe (e.g., which example to load, contacting molecules, sequence information, etc.)
    dataset=PandasDataset(
        id_column="example_id",
        filters=SHARED_TEST_FILTERS,
        data=PN_UNITS_DF,
        name="pn_units",
    ),
    transform=build_rf2aa_transform_pipeline(
        protein_msa_dirs=PROTEIN_MSA_DIRS,
        rna_msa_dirs=RNA_MSA_DIRS,
        n_recycles=5,
        crop_size=256,
        crop_contiguous_probability=1 / 3,
        crop_spatial_probability=2 / 3,
    ),
)

RF2AA_INTERFACES_DATASET = StructuralDatasetWrapper(
    dataset_parser=InterfacesDFParser(),
    dataset=PandasDataset(
        id_column="example_id",
        data=INTERFACES_DF,
        name="interfaces",
        filters=SHARED_TEST_FILTERS,
    ),
    transform=build_rf2aa_transform_pipeline(
        protein_msa_dirs=PROTEIN_MSA_DIRS,
        rna_msa_dirs=RNA_MSA_DIRS,
        n_recycles=5,
        crop_size=256,
        crop_spatial_probability=1.0,
        crop_contiguous_probability=0.0,
    ),
)

# ...build the ConcatDataset
RF2AA_PDB_DATASET = ConcatDatasetWithID(datasets=[RF2AA_PN_UNITS_DATASET, RF2AA_INTERFACES_DATASET])

print("Length of Interfaces Dataset:", len(RF2AA_INTERFACES_DATASET))
print("Length of PN Units Dataset:", len(RF2AA_PN_UNITS_DATASET))
print("Length of PDB Dataset:", len(RF2AA_PDB_DATASET))

### Example: Create a dataset of protein/DNA interactions

In [ ]:
INTERFACES_DF

In [ ]:
# ...get the integer representation of the chain type enum
protein_chain_types = [chain_type.value for chain_type in ChainType.get_proteins()]
dna_chain_type = ChainType.DNA.value

print("Protein chain types:", protein_chain_types)
print("DNA chain type:", dna_chain_type)

In [ ]:
# ...filter the dataframe to only columns that are between a protein and DNA chain
protein_dna_interfaces = INTERFACES_DF[
    (INTERFACES_DF.pn_unit_1_type.isin(protein_chain_types) & (INTERFACES_DF.pn_unit_2_type == dna_chain_type))
    | (INTERFACES_DF.pn_unit_2_type.isin(protein_chain_types) & (INTERFACES_DF.pn_unit_1_type == dna_chain_type))
]
protein_dna_interfaces

## 2.2 Samplers

In [ ]:
from datahub.samplers import calculate_weights_for_pdb_dataset_df
import torch
from torch.utils.data import WeightedRandomSampler

num_examples_per_epoch = 100

# ...calculate the weights based on the AF-3 weighting methodology
b_pn_unit = 0.5  # β_chain
b_interface = 0.5  # β_interface
alphas = {
    "a_prot": 3.0,
    "a_nuc": 3.0,
    "a_ligand": 1.0,
}

pn_units_dataset_weights = calculate_weights_for_pdb_dataset_df(
    dataset_df=RF2AA_PN_UNITS_DATASET._data, alphas=alphas, beta=b_pn_unit
)
interfaces_dataset_weights = calculate_weights_for_pdb_dataset_df(
    dataset_df=RF2AA_INTERFACES_DATASET._data, alphas=alphas, beta=b_interface
)
pdb_dataset_weights = torch.cat([pn_units_dataset_weights, interfaces_dataset_weights])  # NOTE: Order matters!

# ...and initialize one sampler for all PDB datasets, using the unified weights
pdb_sampler = WeightedRandomSampler(
    weights=pdb_dataset_weights,
    num_samples=num_examples_per_epoch,  # We later override with proportional number of examples
    replacement=True,
)

print("PDB Sampler Length:", len(pdb_sampler))
print("PDB Sampler Weights:", pdb_sampler.weights)

### Example: 
Imagine we have the following sampling tree:

        DistributedMixedSampler
                    |
        -------------------------
        |                       |
        0.8                     0.2
        Sampler1              MixedSampler
                            /       \
                            0.9       0.1
                        Sampler2   Sampler3

If we initialized DistributedMixedSampler with `n_examples_per_epoch=100` and `num_replicas=2`, it would collect 80 samples 
from Sampler1 and 20 samples from the MixedSampler. The MixedSampler would in turn collect 18 samples from Sampler2 and 2 samples from Sampler3.
After collecting those 100 samples, the DistributedMixedSampler would shard the samples across the two nodes, ensuring each node receives a unique slice
of 50 examples.

In [ ]:
from torch.utils.data import SequentialSampler
from datahub.samplers import DistributedMixedSampler

# Define a MixedSampler with two samplers
datasets_info = [
    {
        "name": "pdb",
        "dataset": RF2AA_PDB_DATASET,
        "sampler": pdb_sampler,
        "probability": 0.2,
    },
    {
        # NOTE: Illustrative; dataset overlaps with the PDB_DATASET. In actuality, this would be a different dataset (e.g., distillation)
        "name": "pn_units",
        "dataset": RF2AA_PN_UNITS_DATASET,
        "sampler": SequentialSampler(RF2AA_PN_UNITS_DATASET),
        "probability": 0.8,
    },
    # etc.
]
datasets = [dataset_info["dataset"] for dataset_info in datasets_info]

# ...create a sampler that samples from both datasets, according to the probabilities
mixed_sampler = DistributedMixedSampler(
    datasets_info=datasets_info,
    n_examples_per_epoch=100,
    num_replicas=1,
    rank=0,
)

# ...create a dataset including both datasets
concat_dataset = ConcatDatasetWithID(datasets=datasets)

print("Length of PDB Dataset:", len(RF2AA_PDB_DATASET))
print("Length of PN Units Dataset:", len(RF2AA_PN_UNITS_DATASET))
print(
    "Length of Concat Dataset:", len(concat_dataset)
)  # Should be the same as len(RF2AA_PDB_DATASET) + len(RF2AA_PN_UNITS_DATASET)

# ...get the first 10 examples from the mixed sampler
indices = list(mixed_sampler)[:10]
for index in indices:
    print(concat_dataset.get_dataset_by_idx(index))

## 2.3 Transforms

### Setup: Load example from dataframe row

In [ ]:
from datahub.datasets.parsers import load_example_from_metadata_row

# ...grab an example from the PN_UNITS_DF to run through various pipelines
pdb_id = "3ne2"
row = PN_UNITS_DF[PN_UNITS_DF["pdb_id"] == pdb_id].iloc[0]  # Get the first row; we don't care which we choose
original_data = load_example_from_metadata_row(row, PNUnitsDFParser())

# Alternatively, if we wanted to get a single PDB file to run through a pipeline, we could do:
# original_data = parser.parse(filename=Path(f"/projects/ml/frozen_pdb_copies/2024_12_01_pdb/{pdb_id[1:3]}/{pdb_id}.cif.gz"))
# original_data["atom_array"] = original_data["asym_unit"][0] # NOTE: This is the asymmetric unit
# original_data["atom_array"] = original_data["assemblies"]["1"][0] # NOTE: This is the first assembly

In [ ]:
print(original_data.keys())
view(original_data["atom_array"])

### Example 2.3.1: Basic Cropping and Atomization

In [25]:
from copy import deepcopy

data = deepcopy(original_data)

In [ ]:
from datahub.transforms.atom_array import (
    AddGlobalTokenIdAnnotation,
    AddGlobalAtomIdAnnotation,
)
from datahub.transforms.filters import RemoveHydrogens, RemoveTerminalOxygen
from datahub.transforms.atomize import AtomizeByCCDName
from datahub.transforms.crop import CropSpatialLikeAF3
from datahub.transforms.encoding import EncodeAtomArray
from datahub.encoding_definitions import RF2AA_ATOM36_ENCODING
from datahub.transforms.openbabel_utils import AddOpenBabelMoleculesForAtomizedMolecules
from datahub.transforms.base import Compose
from datahub.utils.rng import create_rng_state_from_seeds, rng_state
from datahub.transforms.atom_array import AddWithinPolyResIdxAnnotation

seed = 3
encoding = RF2AA_ATOM36_ENCODING

# We start by performing transforms for basic cropping and atomization of the atom array

basic_crop_and_atomize_pipe = Compose(
    [
        # ...remove hydrogens
        RemoveHydrogens(),
        # ...add a global atom index annotation
        AddGlobalAtomIdAnnotation(),
        # ...add a within-poly residue index annotation (used when indexing into the MSA post-crop)
        AddWithinPolyResIdxAnnotation(),
        # ...remove terminal oxygen atoms
        RemoveTerminalOxygen(),
        # ...atomize residues (e.g., flagging atoms that are part of a ligand)
        AtomizeByCCDName(atomize_by_default=True, res_names_to_ignore=encoding.tokens, move_atomized_part_to_end=True),
        # ...add a global token index annotation
        AddGlobalTokenIdAnnotation(),
        # ...add OpenBabel molecules
        AddOpenBabelMoleculesForAtomizedMolecules(),
        # ...crop
        CropSpatialLikeAF3(crop_size=50, keep_uncropped_atom_array=True),
        # ...encode
        EncodeAtomArray(encoding=encoding),
    ],
    track_rng_state=False,
)

with rng_state(create_rng_state_from_seeds(np_seed=seed, torch_seed=seed, py_seed=seed)):
    data = basic_crop_and_atomize_pipe(data)
    print("Keys after:", data.keys())

In [ ]:
# ...visualize the crop
view(data["atom_array"])

In [ ]:
# ...examine the encoded XYZ coordinates (only of the atoms within the crop)
# (They are encoded according to the `RF2AA_ATOM36_ENCODING` encoding)
print(list(data["encoded"].keys()))
print(data["encoded"]["xyz"].shape)
print(data["encoded"]["xyz"])

In [ ]:
# If we wanted to run the pipeline again, with the SAME random seeds, we can:
with rng_state(create_rng_state_from_seeds(np_seed=seed, torch_seed=seed, py_seed=seed)):
    data = basic_crop_and_atomize_pipe(deepcopy(original_data))

# ..and we can see that the crop is the same
view(data["atom_array"])

In [ ]:
# If we didn't set the seed, the crop would be different
data = basic_crop_and_atomize_pipe(deepcopy(original_data))

# ..and we can see that the crop is different
view(data["atom_array"])

In [ ]:
# We also have functional API's for many transforms, if we don't need the full Transform capabilities

from datahub.transforms.crop import crop_spatial_like_af3
from datahub.transforms.atom_array import add_global_atom_id_annotation
from datahub.transforms.atomize import atomize_by_ccd_name

atom_array = deepcopy(original_data["atom_array"])

atom_array = add_global_atom_id_annotation(atom_array)

atom_array = atomize_by_ccd_name(atom_array, atomize_by_default=False)

crop_info_dict = crop_spatial_like_af3(
    atom_array=atom_array,
    crop_size=50,
    query_pn_unit_iids=np.unique(atom_array.pn_unit_iid),
)
atom_array = atom_array[crop_info_dict["crop_atom_idxs"]]

view(atom_array)

### Example 2.3.2: MSAs

In [34]:
from datahub.transforms.msa.msa import (
    EncodeMSA,
    FillFullMSAFromEncoded,
    LoadPolymerMSAs,
    PairAndMergePolymerMSAs,
)

PAD_TOKEN = encoding.token_to_idx["UNK"]

# We continue by loading, pairing, merging, and encoding MSAs
# By performing these transforms post-crop, we only process MSAs for proteins that are present in the crop

msa_pipe = Compose(
    [
        LoadPolymerMSAs(protein_msa_dirs=PROTEIN_MSA_DIRS, rna_msa_dirs=RNA_MSA_DIRS, max_msa_sequences=100),
        PairAndMergePolymerMSAs(),
        EncodeAtomArray(encoding),
        # MSA featurize workflow
        EncodeMSA(encoding=encoding, token_to_use_for_gap=PAD_TOKEN),
        FillFullMSAFromEncoded(pad_token=PAD_TOKEN),
    ],
    track_rng_state=False,
)

output = msa_pipe(data)

In [ ]:
# ...print the keys of the resulting dictionary
print("Keys in output dictionary:", output.keys())

# ...examine the encoded MSA details
print("Keys in polymer_msas_by_chain_id:", output["polymer_msas_by_chain_id"]["C"].keys())

# ...inspect the MSAs (PRIOR to cropping)
print(output["polymer_msas_by_chain_id"]["C"]["encoded_msa"].shape)
print(output["polymer_msas_by_chain_id"]["C"]["encoded_msa"])

### Example 2.3.4: Pipeline Checks

In [ ]:
# Only some ordering of Transforms are valid
# If we try and perform Transforms in an incompatible order (e.g., AtomizeByCCDName after CropSpatialLikeAF3), we will throw a Transform pipeline error

pipe_that_will_fail = Compose(
    [
        AtomizeByCCDName(atomize_by_default=True, res_names_to_ignore=encoding.tokens, move_atomized_part_to_end=True),
    ]
)

data = pipe_that_will_fail(data)

In [ ]:
# If we want to see which transforms we've run, we can print the transform history
print(data.__transform_history__)

### Example 2.3.3: Full AF-3 Pipeline

#### Build the pipeline

In [43]:
from datahub.transforms.af3_reference_molecule import GetAF3ReferenceMoleculeFeatures
from datahub.transforms.atom_array import (
    AddGlobalAtomIdAnnotation,
    AddWithinChainInstanceResIdx,
    AddWithinPolyResIdxAnnotation,
    AddGlobalTokenIdAnnotation,
)
from datahub.transforms.filters import HandleUndesiredResTokens, RemoveHydrogens, RemoveUnresolvedPNUnits
from datahub.transforms.atomize import AtomizeByCCDName, FlagNonPolymersForAtomization
from datahub.transforms.base import Compose, ConvertToTorch, RandomRoute, SubsetToKeys
from datahub.transforms.covalent_modifications import FlagAndReassignCovalentModifications
from datahub.transforms.crop import CropContiguousLikeAF3, CropSpatialLikeAF3
from datahub.transforms.encoding import EncodeAF3TokenLevelFeatures
from datahub.transforms.msa.msa import (
    EncodeMSA,
    FeaturizeMSALikeAF3,
    FillFullMSAFromEncoded,
    LoadPolymerMSAs,
    PairAndMergePolymerMSAs,
)
from datahub.transforms.template import AddRFTemplates, FeaturizeTemplatesLikeAF3
from cifutils.constants import AF3_EXCLUDED_LIGANDS, STANDARD_AA, STANDARD_DNA, STANDARD_RNA
from datahub.encoding_definitions import AF3SequenceEncoding

af3_sequence_encoding = AF3SequenceEncoding()

# ...transforms pre-cropping
transforms = [
    RemoveHydrogens(),
    RemoveUnresolvedPNUnits(),  # Remove PN units that are unresolved early (and also after cropping)
    HandleUndesiredResTokens(AF3_EXCLUDED_LIGANDS),  # e.g., non-standard residues
    FlagAndReassignCovalentModifications(),
    FlagNonPolymersForAtomization(),
    AddGlobalAtomIdAnnotation(),
    AtomizeByCCDName(
        atomize_by_default=True,
        res_names_to_ignore=STANDARD_AA + STANDARD_RNA + STANDARD_DNA,
        move_atomized_part_to_end=False,
        validate_atomize=False,
    ),
    AddWithinChainInstanceResIdx(),
    AddWithinPolyResIdxAnnotation(),
]

# ...crop
crop_size = 256
crop_center_cutoff_distance = 10.0
crop_contiguous_probability = 1 / 3
crop_spatial_probability = 2 / 3

contiguous_crop_transform = CropContiguousLikeAF3(crop_size=256, keep_uncropped_atom_array=True)
spatial_crop_transform = CropSpatialLikeAF3(
    crop_size=crop_size, crop_center_cutoff_distance=crop_center_cutoff_distance, keep_uncropped_atom_array=True
)
if crop_contiguous_probability > 0 and crop_spatial_probability > 0:
    transforms += [
        # ...crop around our query pn_unit(s) early, since we don't need the full structure moving forward
        RandomRoute(
            transforms=[
                contiguous_crop_transform,
                spatial_crop_transform,
            ],
            probs=[crop_contiguous_probability, crop_spatial_probability],
        ),
    ]
elif crop_contiguous_probability > 0:
    transforms.append(contiguous_crop_transform)
elif crop_spatial_probability > 0:
    transforms.append(spatial_crop_transform)

# ...templates, token-level features
transforms += [
    EncodeAF3TokenLevelFeatures(sequence_encoding=af3_sequence_encoding),
    AddGlobalTokenIdAnnotation(),
    GetAF3ReferenceMoleculeFeatures(),
    AddRFTemplates(),
    FeaturizeTemplatesLikeAF3(
        sequence_encoding=af3_sequence_encoding,
    ),
]

# ...MSAs
transforms += [
    # ...load and pair MSAs
    LoadPolymerMSAs(),
    PairAndMergePolymerMSAs(dense=True),
    # ...encode MSA to AF-3 format
    EncodeMSA(encoding=af3_sequence_encoding, token_to_use_for_gap=af3_sequence_encoding.token_to_idx["<G>"]),
    # ...fill MSA, indexing into only the portions of the polymers that are present in the cropped structure
    FillFullMSAFromEncoded(pad_token=af3_sequence_encoding.token_to_idx["<G>"]),
    # ...featurize MSA (we need to convert form numpy to torch first, since MSA Featurize expects torch tensors)
    ConvertToTorch(
        keys=[
            "encoded",
            "feats",
            "full_msa_details",
        ]
    ),
    FeaturizeMSALikeAF3(
        encoding=af3_sequence_encoding,
        n_recycles=5,
        n_msa=10,
    ),
    # ... remove all non-feature keys (to make compatible with generic batch_collate, which only allows tensors, numpy arrays, str, etc.)
    SubsetToKeys(["example_id", "feats"]),
]

# ... compose final pipeline
af3_pipe = Compose(transforms)

#### Run the pipeline

In [ ]:
data = deepcopy(original_data)

# ...ensure that we have a clean copy of the original data (no transforms have been applied)
print(data.keys())

In [ ]:
# ...apply the AF3 pipeline
af3_features = af3_pipe(data)

In [ ]:
# ...print the keys of the resulting dictionary
print(af3_features.keys())

# ...print the keys within the features dictionary
print(af3_features["feats"].keys())

In [ ]:
af3_features["feats"]["token_index"]

# 3. `Datahub` - as a DEVELOPER

Imagine that you're building a custom data pipeline that requires transformations beyond those provided in the base `datahub` repository...

In [ ]:
# Write a new transform that does something..
from datahub.transforms.base import Transform
#  ^--- this is the base class for all transforms in datahub, we will need to subclass this

from datahub.transforms._checks import check_contains_keys
#  ^--- `_checks` has helper functions for ensuring the pre-conditions for running a transform are met

import scipy

# If you like a functional interface, you can implement everything that goes into the forward pass in a
#  separate function, and call that function from the forward method


def calculate_traversal_distance_matrix(rf2aa_bond_features_matrix: np.ndarray) -> np.ndarray:
    # RF2AA uses the following bond mapping, as described in ChemData:
    #     - 0 = No bonds
    #     - 1 = Single bond
    #     - 2 = Double bond
    #     - 3 = Triple bond
    #     - 4 = Aromatic

    # Reduce the bond features matrix to only include atom-atom bonds
    atom_bonds = (rf2aa_bond_features_matrix > 0) * (rf2aa_bond_features_matrix < 5)

    # Compute the shortest path distance matrix using scipy
    traversal_distance_matrix = scipy.sparse.csgraph.shortest_path(atom_bonds, directed=False)

    return traversal_distance_matrix


class AddRF2AATraversalDistanceMatrix(Transform):
    """
    Generates a matrix indicating the minimum amount of bonds to traverse between two nodes.
    We define the traversal distance between two protein nodes as zero.
    Sets the "traversal_distance_matrix" key in the data dictionary.

    From the RF2AA supplement, Supplementary Methods Table 8: Inputs to RFAA:
    ------------------------------------------------------------------------------------------------
    dist_matrix               | (L, L) Minimum amount of bonds to traverse between two nodes.
                                This is 0 between all protein nodes.
    ------------------------------------------------------------------------------------------------
    """

    # requires_previous_transforms = []  # <-- not used here, but this can be used to ensure certain transforms are run prior to this one
    # previous_transform_order_matters = False  # <-- not used here, but this can be used to ensure the previous transforms are run in a specific order
    # incompatible_with_previous_transforms = []  # <-- not used here, but this can be used to ensure no incompatible transforms are run prior to this one

    def check_input(self, data: dict):
        check_contains_keys(data, ["rf2aa_bond_features_matrix"])

    def forward(self, data: dict) -> dict:
        rf2aa_bond_features_matrix = data["rf2aa_bond_features_matrix"]

        # Add to the data dictionary
        # NOTE: This matrix will have infinity values, which are handled downstream by the model
        data["rf2aa_traversal_distance_matrix"] = calculate_traversal_distance_matrix(rf2aa_bond_features_matrix)

        # finally, return the updated data dictionary
        # !IMPORTANT! All `Transforms` must return the data dictionary in their `forward` method
        return data


dummy_data = {"rf2aa_bond_features_matrix": np.random.randint(0, 5, size=(10, 10))}

transform = AddRF2AATraversalDistanceMatrix()

dummy_data = transform(dummy_data)

# Print the new key that was added to the data dictionary
print(dummy_data["rf2aa_traversal_distance_matrix"])

# See if the transform was added to the transform history
# ... this automatically profiles your transforms as you go
print(dummy_data.__transform_history__)

We can now compose this transform with any others to form full fledged data pipelines. Let's do a simple one here:

In [ ]:
from datahub.transforms.base import Compose, RemoveKeys

transform = Compose(
    [
        # ... compute and add the traversal distance matrix
        AddRF2AATraversalDistanceMatrix(),
        # ... remove the original bond features matrix, since we don't need it anymore
        RemoveKeys(["rf2aa_bond_features_matrix"]),
    ]
)

dummy_data = {"rf2aa_bond_features_matrix": np.random.randint(0, 5, size=(10, 10))}
print("Keys in original data:", dummy_data.keys())
dummy_data = transform(dummy_data)

print("Keys in transformed data:", dummy_data.keys())

### Debugging

In [ ]:
from datahub.transforms.base import RaiseOnCondition

pipe = Compose(
    [
        # ... compute and add the traversal distance matrix
        AddRF2AATraversalDistanceMatrix(),
        # ... remove the original bond features matrix, since we don't need it anymore
        RemoveKeys(["rf2aa_bond_features_matrix"]),
        # ... raise an error if the traversal distance matrix is not finite
        RaiseOnCondition(condition=lambda data: True, error_message="YOU SHALL NOT PASS!"),
    ],
    print_rng_state=True,
)

dummy_data = {"rf2aa_bond_features_matrix": np.random.randint(0, 5, size=(10, 10)), "example_id": "my_broken_example"}

dummy_data = pipe(dummy_data)

If you wanted to step up until that position in the pipeline but not use a debugger for some reason (e.g. because you prefer to run interactively in a notebook for development), the `Compose` object provides
a hiddend `_stop_before` argument for you.

In [ ]:
# all good untils here -- phew!
pipe(dummy_data, _stop_before="RaiseOnCondition")